# Tensorflow2.0 小练习

本 notebook 用来练习 **基础函数实现**（softmax / sigmoid / 交叉熵）。

- **函数定义位置**：下面 4 个 code cell（softmax、sigmoid、softmax_ce、sigmoid_ce）。
- **拟合效果/验证方式**：每个函数后面的测试表达式会与 TensorFlow 内置实现做数值对齐。


In [ ]:
import tensorflow as tf
import numpy as np

## 实现softmax函数

In [ ]:
def softmax(x):
    ##########
    '''实现 softmax 函数，只要求对最后一维归一化。
    约束：不允许用 tf 自带的 softmax（tf.nn.softmax）。'''
    ##########
    x = tf.convert_to_tensor(x)
    # 数值稳定性：对最后一维减去 max，再做 exp
    x = x - tf.reduce_max(x, axis=-1, keepdims=True)
    exp_x = tf.exp(x)
    prob_x = exp_x / tf.reduce_sum(exp_x, axis=-1, keepdims=True)
    return prob_x

test_data = np.random.normal(size=[10, 5])
(softmax(test_data).numpy() - tf.nn.softmax(test_data, axis=-1).numpy())**2 < 1e-4

## 实现sigmoid函数

In [ ]:
def sigmoid(x):
    ##########
    '''实现 sigmoid 函数。
    约束：不允许用 tf 自带的 sigmoid（tf.nn.sigmoid）。'''
    ##########
    x = tf.convert_to_tensor(x)
    prob_x = 1.0 / (1.0 + tf.exp(-x))
    return prob_x

test_data = np.random.normal(size=[10, 5])
(sigmoid(test_data).numpy() - tf.nn.sigmoid(test_data).numpy())**2 < 0.0001

## 实现 softmax 交叉熵loss函数

In [ ]:
def softmax_ce(x, label):
    ##########
    '''实现 softmax 交叉熵 loss（输入 x 为 softmax 概率）。
    约束：不允许用 tf 自带的 softmax_cross_entropy。'''
    ##########
    x = tf.convert_to_tensor(x)
    label = tf.cast(label, x.dtype)
    eps = tf.constant(1e-12, dtype=x.dtype)
    # CE = - sum_i y_i * log(p_i)
    loss = -tf.reduce_mean(tf.reduce_sum(label * tf.math.log(x + eps), axis=-1))
    return loss

test_data = np.random.normal(size=[10, 5])
prob = tf.nn.softmax(test_data)
label = np.zeros_like(test_data)
label[np.arange(10), np.random.randint(0, 5, size=10)]=1.

((tf.reduce_mean(tf.nn.softmax_cross_entropy_with_logits(label, test_data))
  - softmax_ce(prob, label))**2 < 0.0001).numpy()

## 实现 sigmoid 交叉熵loss函数

In [ ]:
def sigmoid_ce(x, label):
    ##########
    '''实现 sigmoid 交叉熵 loss（输入 x 为 sigmoid 概率）。
    约束：不允许用 tf 自带的 sigmoid_cross_entropy。'''
    ##########
    x = tf.convert_to_tensor(x)
    label = tf.cast(label, x.dtype)
    eps = tf.constant(1e-12, dtype=x.dtype)
    # BCE = -[ y*log(p) + (1-y)*log(1-p) ]
    loss = -tf.reduce_mean(label * tf.math.log(x + eps) + (1.0 - label) * tf.math.log(1.0 - x + eps))
    return loss

test_data = np.random.normal(size=[10])
prob = tf.nn.sigmoid(test_data)
label = np.random.randint(0, 2, 10).astype(test_data.dtype)
print (label)

((tf.reduce_mean(tf.nn.sigmoid_cross_entropy_with_logits(label, test_data))- sigmoid_ce(prob, label))**2 < 0.0001).numpy()
